# Resolución de los ejercicios – Correlación y Multicolinealidad

Este notebook contiene las soluciones completas de los ejercicios propuestos a los alumnos.  
Se recomienda usarlo solo después de que los estudiantes hayan intentado resolver por sí mismos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

sns.set_theme(style="whitegrid")
np.random.seed(123)

## Ejercicio 1 – Cálculo manual de Pearson

In [ ]:
x = np.array([2, 4, 6, 8, 10], dtype=float)
y = np.array([3, 7, 5, 11, 9], dtype=float)

media_x = x.mean()
media_y = y.mean()
print(f"Media x = {media_x}, Media y = {media_y}")

dx = x - media_x
dy = y - media_y

n = len(x)
cov_xy = np.sum(dx * dy) / (n - 1)
sx = np.sqrt(np.sum(dx**2) / (n - 1))
sy = np.sqrt(np.sum(dy**2) / (n - 1))

r_manual = cov_xy / (sx * sy)
print(f"Covarianza = {cov_xy:.4f}")
print(f"sx = {sx:.4f}, sy = {sy:.4f}")
print(f"r manual = {r_manual:.4f}")

r_scipy, p = stats.pearsonr(x, y)
print(f"r scipy  = {r_scipy:.4f} (p = {p:.4f})")

**Nota**: el valor exacto depende de si se usa covarianza muestral (n-1) o poblacional (n). `scipy.stats.pearsonr` usa la versión con n-1 en el denominador de la covarianza, por lo que debe coincidir.

## Ejercicio 2 – Comparar los tres coeficientes (relación monotónica no lineal)

In [ ]:
n = 150
x = np.random.uniform(-3, 3, n)
y = x**3 + np.random.normal(0, 2, n)

r_p, _ = stats.pearsonr(x, y)
r_s, _ = stats.spearmanr(x, y)
r_k, _ = stats.kendalltau(x, y)

print(f"Pearson : {r_p:.3f}")
print(f"Spearman: {r_s:.3f}")
print(f"Kendall : {r_k:.3f}")

plt.figure(figsize=(7, 5))
plt.scatter(x, y, alpha=0.7, edgecolor="none")
plt.title("Relación monotónica no lineal ($y = x^3$ + ruido)")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

**Interpretación**  
Spearman y Kendall capturan mejor la relación porque es monotónica (aunque no lineal). Pearson solo mide la componente lineal y, por tanto, suele dar un valor más bajo. Kendall normalmente es numéricamente menor que Spearman, pero ambos apuntan en la misma dirección.

## Ejercicio 3 – Matriz de correlación y redundancias

In [ ]:
np.random.seed(7)
n = 250
precio = np.random.uniform(10, 100, n)
descuento = precio * 0.15 + np.random.normal(0, 3, n)
unidades = 200 - 1.2 * precio + 0.8 * descuento + np.random.normal(0, 15, n)
publicidad = np.random.uniform(5, 50, n)
satisfaccion = 0.05 * unidades + 0.1 * publicidad + np.random.normal(0, 5, n)

df_tienda = pd.DataFrame({
    "precio": precio,
    "descuento": descuento,
    "unidades": unidades,
    "publicidad": publicidad,
    "satisfaccion": satisfaccion
})

corr = df_tienda.corr(method="pearson")
print(corr.round(3))

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True)
plt.title("Matriz de correlación (Pearson)")
plt.tight_layout()
plt.show()

# Par con mayor |correlación| fuera de la diagonal
corr_abs = corr.abs()
np.fill_diagonal(corr_abs.values, 0)
max_pair = corr_abs.unstack().sort_values(ascending=False).head(1)
print("\nPar más correlacionado (en valor absoluto):")
print(max_pair)

**Respuesta esperada**  
El par `precio` – `descuento` suele tener la correlación más alta (por construcción ~0.9).  
En un modelo de regresión, incluir ambas variables generaría multicolinealidad. Una de las dos (normalmente `descuento`, si se deriva del precio) puede considerarse redundante o debe tratarse con cuidado (centrado, regularización, etc.).

## Ejercicio 4 – VIF

In [ ]:
def calcular_vif(dataframe):
    X = add_constant(dataframe)
    vif_data = pd.DataFrame({
        "variable": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })
    return vif_data.query("variable != 'const'").sort_values("VIF", ascending=False)

print("VIF del dataset completo:")
print(calcular_vif(df_tienda).round(2))

print("\nVIF tras eliminar 'descuento':")
print(calcular_vif(df_tienda.drop(columns=["descuento"])).round(2))

**Interpretación**  
- Con todas las variables, `precio` y `descuento` suelen mostrar VIF elevados (fácilmente > 5 o > 10).  
- Al eliminar `descuento`, los VIF de las variables restantes bajan a valores aceptables (cerca de 1–3).  
- Esto confirma que la multicolinealidad provenía principalmente de la relación entre precio y descuento.

## Ejercicio 5 – Relación no monotónica (forma de U)

In [ ]:
n = 200
x = np.random.uniform(0, 10, n)
y = (x - 5)**2 + np.random.normal(0, 2, n)

r_p, _ = stats.pearsonr(x, y)
r_s, _ = stats.spearmanr(x, y)
r_k, _ = stats.kendalltau(x, y)

print(f"Pearson : {r_p:.3f}")
print(f"Spearman: {r_s:.3f}")
print(f"Kendall : {r_k:.3f}")

plt.figure(figsize=(7, 5))
plt.scatter(x, y, alpha=0.7, edgecolor="none")
plt.title("Relación no monotónica (forma de U)")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

**Respuestas**  
1. Los tres coeficientes suelen ser cercanos a cero (o muy bajos).  
2. Ninguno refleja bien la fuerza de la asociación, porque la relación no es monotónica.  
3. En la práctica: **siempre visualizar** (scatter plots, residuales, LOWESS, etc.). También se pueden usar medidas de dependencia más generales (información mutua, distancia de correlación, etc.) o transformar / modelar de forma no lineal.

## Ejercicio 6 (bonus) – Efecto de un outlier

In [ ]:
np.random.seed(42)
n = 100
x = np.random.normal(50, 10, n)
y = 1.5 * x + np.random.normal(0, 8, n)

r_p_antes, _ = stats.pearsonr(x, y)
r_s_antes, _ = stats.spearmanr(x, y)
print("Antes del outlier:")
print(f"  Pearson  = {r_p_antes:.3f}")
print(f"  Spearman = {r_s_antes:.3f}")

# Añadir un outlier extremo
x_out = np.append(x, 120)
y_out = np.append(y, 20)   # valor muy bajo respecto a la tendencia

r_p_despues, _ = stats.pearsonr(x_out, y_out)
r_s_despues, _ = stats.spearmanr(x_out, y_out)
print("\nDespués del outlier:")
print(f"  Pearson  = {r_p_despues:.3f}")
print(f"  Spearman = {r_s_despues:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(x, y, alpha=0.7)
axes[0].set_title("Sin outlier")
axes[1].scatter(x_out, y_out, alpha=0.7)
axes[1].scatter([120], [20], color="red", s=80, label="outlier")
axes[1].legend()
axes[1].set_title("Con outlier")
plt.tight_layout()
plt.show()

**Comentario**  
Pearson suele verse mucho más afectado por el outlier (el valor de r puede bajar de forma notable). Spearman, al basarse en rangos, es considerablemente más robusto: el cambio es mucho menor. Esta es una de las razones principales para preferir Spearman (o Kendall) cuando se sospecha la presencia de valores extremos.